# 🧪 tiny-distill — Train Your Own BitNet Model

This notebook lets you train a tiny 1.58-bit model from your collected data.

**Free GPU:** Google Colab gives you free T4 GPU (16GB VRAM) for ~4 hours.

## Steps:
1. Upload your collected data (JSONL file)
2. Run all cells
3. Download your trained model

In [ ]:
# 1️⃣ Setup — Install dependencies
!pip install -q torch transformers datasets accelerate peft bitsandbytes
print('✅ Dependencies installed')

In [ ]:
# 2️⃣ Clone tiny-distill
!git clone https://github.com/YOUR_USERNAME/tiny-distill.git
%cd tiny-distill
print('✅ Repo cloned')

In [ ]:
# 3️⃣ Upload your training data
# Option A: Upload manually
from google.colab import files
uploaded = files.upload()  # Upload your .jsonl file
DATA_FILE = list(uploaded.keys())[0]
print(f'✅ Uploaded: {DATA_FILE}')

# Option B: Use example data (for testing)
# DATA_FILE = 'data/example_dataset.jsonl'

In [ ]:
# 4️⃣ Verify data
import json
with open(DATA_FILE) as f:
    lines = f.readlines()
print(f'📊 Dataset: {len(lines)} examples')
print(f'\nSample:')
sample = json.loads(lines[0])
print(f'  Prompt: {sample["prompt"][:100]}...')
print(f'  Response: {sample["response"][:100]}...')

In [ ]:
# 5️⃣ Check GPU
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'🎮 GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️ No GPU — go to Runtime > Change runtime type > GPU')

In [ ]:
# 6️⃣ Train!
# This fine-tunes Microsoft's BitNet 2B model on your data
# Using LoRA to fit in Colab's free T4 (16GB VRAM)

!python train/train_bitnet.py \
    --data {DATA_FILE} \
    --base-model microsoft/bitnet-b1.58-2B-4T-bf16 \
    --output models/my-distilled-model \
    --epochs 3 \
    --batch-size 4 \
    --lr 2e-5 \
    --max-seq-length 1024 \
    --lora-r 16

In [ ]:
# 7️⃣ Test your model
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_path = 'models/my-distilled-model'
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map='auto'
)

# Test prompt
test_prompt = 'Explain quantum computing in simple terms.'
inputs = tokenizer(test_prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        do_sample=True
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f'\n🧪 Prompt: {test_prompt}')
print(f'\n📝 Response:\n{response}')

In [ ]:
# 8️⃣ Download your model
import shutil
shutil.make_archive('my-distilled-model', 'zip', 'models/my-distilled-model')
files.download('my-distilled-model.zip')
print('✅ Model downloaded!')

## 🎉 Next Steps

1. **Run locally:** Convert to GGUF and use with [bitnet.cpp](https://github.com/microsoft/BitNet) for CPU inference
2. **Upload to Hugging Face:** Share your model with the world
3. **Collect more data:** More diverse data = better model
4. **Try different base models:** Qwen, Gemma, Phi also work